In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
)

In [4]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)
df.describe()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
count,4631.000000,5000.000000,5000.000000,4965.000000,5000.000000
mean,54533.212049,2.089400,4.472600,0.494767,0.576600
std,21245.661710,1.188987,2.435906,0.199431,0.494147
min,12000.000000,0.000000,0.000000,0.010000,0.000000
25%,40458.500000,1.000000,3.000000,0.360000,0.000000
50%,55051.000000,2.000000,4.000000,0.490000,1.000000
75%,71252.500000,3.000000,6.000000,0.630000,1.000000
max,109886.000000,6.000000,13.000000,0.990000,1.000000


In [5]:
print("Missing values before filling:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Missing values before filling:
lead_source          148
industry             240
employment_status    194
location             208
annual_income        369
lead_score            35
dtype: int64


In [6]:
for col in df.columns:
    if col == "converted":
        continue
    if df[col].dtype == "object" or isinstance(
        df[col].dtype, pd.CategoricalDtype
    ):
        df[col] = df[col].fillna("NA")
    else:
        df[col] = df[col].fillna(0.0)

In [7]:
df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=1
)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

target = "converted"

def get_xy(frame):
    X = frame.drop(columns=[target])
    y = frame[target].astype(int)
    return X, y

X_train, y_train = get_xy(df_train)
X_val, y_val = get_xy(df_val)

```
1. ROC AUC feature importance (1 point)


lead_score

number_of_courses_viewed

interaction_count

annual_income
```

In [9]:
for col in [
    "lead_score",
    "number_of_courses_viewed",
    "interaction_count",
    "annual_income",
]:
    auc = roc_auc_score(y_train, X_train[col])
    if auc < 0.5:
        auc = roc_auc_score(y_train, -X_train[col])
    print(f"{col}: {auc:.3f}")

lead_score: 0.788
number_of_courses_viewed: 0.723
interaction_count: 0.765
annual_income: 0.608


```
2. Model AUC (1 point)


0.532

0.632

0.732

0.832
```

In [15]:
X_train_encoded = X_train_encoded.tocsr()
X_val_encoded = X_val_encoded.tocsr()

for X in [X_train_encoded, X_val_encoded]:
    X.indices = X.indices.astype(np.int32)
    X.indptr = X.indptr.astype(np.int32)

model = make_model(C=1.0)
model.fit(X_train_encoded, y_train)

y_pred = model.predict_proba(X_val_encoded)[:, 1]
print(round(roc_auc_score(y_val, y_pred), 3))

0.732


```
3. Precision and recall (1 point)


0.43

0.63

0.73

0.83
```

In [16]:
thresholds = np.arange(0.0, 1.01, 0.01)
best_threshold = None
best_diff = float("inf")

for t in thresholds:
    y_pred_binary = (y_pred >= t).astype(int)

    p = precision_score(y_val, y_pred_binary, zero_division=0)
    r = recall_score(y_val, y_pred_binary, zero_division=0)

    if p == 0 and r == 0:
        continue

    diff = abs(p - r)

    if diff < best_diff:
        best_diff = diff
        best_threshold = t

print("Best threshold:", round(best_threshold, 2))

Best threshold: 0.63


```
4. F1 score (1 point)


0.21

0.41

0.61

0.81
```


In [18]:
best_f1 = 0
best_threshold = 0

for t in np.arange(0.0, 1.01, 0.01):
    y_pred_binary = (y_pred >= t).astype(int)

    p = precision_score(y_val, y_pred_binary, zero_division=0)
    r = recall_score(y_val, y_pred_binary, zero_division=0)

    f1 = 0 if p + r == 0 else 2 * p * r / (p + r)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print("Best threshold:", round(best_threshold, 2))
print("Maximum F1:", round(best_f1, 3))

Best threshold: 0.41
Maximum F1: 0.758


```
5. 5-Fold CV standard deviation (1 point)


0.001

0.007

0.013

0.060
```

In [20]:
def cross_validate_auc(C):
    df = df_full_train.copy()
    y = df["converted"].astype(int)
    X = df.drop(columns=["converted"])

    kfold = KFold(n_splits=5, shuffle=True, random_state=1)
    scores = []

    for train_idx, val_idx in kfold.split(X):
        X_train_fold = X.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        dv = DictVectorizer()
        X_train_dict = X_train_fold.to_dict(orient="records")
        X_val_dict = X_val_fold.to_dict(orient="records")

        X_train_encoded = dv.fit_transform(X_train_dict)
        X_val_encoded = dv.transform(X_val_dict)

        # Work around sparse-index compatibility issues
        for matrix in [X_train_encoded, X_val_encoded]:
            matrix.indices = matrix.indices.astype(np.int32)
            matrix.indptr = matrix.indptr.astype(np.int32)

        model = LogisticRegression(
            solver="liblinear", C=C, max_iter=1000
        )
        model.fit(X_train_encoded, y_train_fold)

        y_pred_fold = model.predict_proba(X_val_encoded)[:, 1]
        auc = roc_auc_score(y_val_fold, y_pred_fold)
        scores.append(auc)

    return np.array(scores)

In [21]:
scores = cross_validate_auc(C=1.0)

print("Fold AUC scores:", scores)
print("Mean AUC:", round(scores.mean(), 3))
print("Standard deviation:", round(scores.std(), 3))

Fold AUC scores: [0.73053442 0.72974415 0.73634924 0.74130297 0.72104304]
Mean AUC: 0.732
Standard deviation: 0.007


```
6. Best C (1 point)


0.000001

0.001

1
```

In [25]:
for C in [0.000001, 0.001, 1]:
    scores = cross_validate_auc(C=C)
    print(
        f"C={C}: mean AUC={scores.mean():.3f}, "
        f"std={scores.std():.3f}"
    )

C=1e-06: mean AUC=0.617, std=0.019
C=0.001: mean AUC=0.749, std=0.010
C=1: mean AUC=0.732, std=0.007
